In [8]:
import kagglehub


path = kagglehub.dataset_download("alessiocorrado99/animals10")

print("D:/Acads/4th year/ML/ML_Project_FL/Federated-machine-learning", path)

D:/Acads/4th year/ML/ML_Project_FL/Federated-machine-learning C:\Users\varad\.cache\kagglehub\datasets\alessiocorrado99\animals10\versions\2


In [1]:
import torch
from torch.utils.data import random_split, Subset
import os
import data

from client import FederatedClient
from server import Server
from data import Centralized, data_store  

def prepare_client_datasets(full_dataset, num_clients: int = 5, samples_per_client: int = 20):
    """
    Randomly split the full dataset into smaller datasets for each client.

    Args:
        full_dataset: Complete torch Dataset.
        num_clients: Number of clients.
        samples_per_client: Number of images per client.

    Returns:
        List of torch.utils.data.Dataset, one for each client.
    """
    total_needed = num_clients * samples_per_client
    assert total_needed <= len(full_dataset), "Not enough samples in dataset."

    indices = torch.randperm(len(full_dataset))[:total_needed].tolist()
    selected_subset = Subset(full_dataset, indices)

    client_datasets = random_split(selected_subset, [samples_per_client] * num_clients)
    return client_datasets

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    dataset_path = os.path.join('dataset', 'raw-img')

    
    full_dataset = data_store(dataset_path)
    print(f"Total images loaded: {len(full_dataset)}")

    num_clients = 5
    samples_per_client = 20
    client_datasets = prepare_client_datasets(full_dataset, num_clients, samples_per_client)

    
    base_model = Centralized()

    
    clients = []
    for idx in range(num_clients):
        client = FederatedClient(
            data=client_datasets[idx],
            model=Centralized(),  # Fresh model for each client
            device=device
        )
        clients.append(client)

    
    server = Server(model=base_model.to(device), client_list=clients)

    
    num_rounds = 3
    for round_num in range(num_rounds):
        print(f"\n--- Federated Round {round_num+1} ---")
        server.federated_round(num_selected_clients=3, epochs=2, lr=0.01)

    
    print("\n--- Final Evaluation ---")
    
    test_loader = torch.utils.data.DataLoader(full_dataset, batch_size=32, shuffle=False)
    server.evaluate_global_model(test_loader)

if __name__ == "__main__":
    main()


Using device: cpu
Total images loaded: 26179

--- Federated Round 1 ---
[Client 0] Loss: 2.2853, Samples: 40
[Client 1] Loss: 2.2923, Samples: 40

--- Federated Round 2 ---
[Client 0] Loss: 2.2728, Samples: 40
[Client 1] Loss: 2.2876, Samples: 40

--- Federated Round 3 ---
[Client 0] Loss: 2.2599, Samples: 40
[Client 1] Loss: 2.2824, Samples: 40

--- Final Evaluation ---
Global model accuracy: 18.42%


In [2]:
import matplotlib.pyplot as plt


def show_image(image_array):
    
    image_array = image_array.transpose(1, 2, 0)  
    image_array = np.clip(image_array, 0, 1)       
    plt.imshow(image_array)
    plt.axis('off')


def visualize_client_data(client, num_images=5):
    plt.figure(figsize=(15, 3))
    for idx in range(min(num_images, len(client.data))):
        plt.subplot(1, num_images, idx + 1)
        show_image(client.data[idx])
        plt.title(f"Label: {client.labels[idx]}")
    plt.show()


if __name__ == "__main__":
    clients = create_clients_from_animals(num_clients=10)

    
    visualize_client_data(clients[0], num_images=5)


FileNotFoundError: [WinError 3] The system cannot find the path specified: 'raw-img\\cat'